# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from groq import Groq

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    raise ValueError("GROQ_API_KEY was not found. Add it to your .env file.")
elif api_key.strip() != api_key:
    raise ValueError("GROQ_API_KEY has leading or trailing whitespace; remove it from your .env file.")
else:
    print("Groq API key found.")
    
MODEL = 'openai/gpt-oss-120b'
groq = Groq(api_key=api_key)

Groq API key found.


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have Groq figure out which links are relevant

### Use a Groq call to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [9]:
def select_relevant_links(url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt for Groq

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 10 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
LiquidAI/LFM3-1.7B
Updated
5 days ago
•
42.4k
•
925
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
17 days ago
•
177k
•
4.66k
nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B
Updated
4 days ago
•
32.4k
•
665
mistralai/Mistral-Large-4-1T-A52B
Updated
about 3 hours ago
•
5.5k
•
1.49k
baidu/Unlimited-OCR-2
Updated
about 12 hours ago
•
148k
•
553
Browse 2M+ models
Spaces
Running
on
RTX PRO 600

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nLiquidAI/LFM3-1.7B\nUpdated\n5 days ago\n•\n42.4k\n•\n926\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated\n17 days ago\n•\

In [35]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    return result

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 10 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your gateway to open‑source machine‑learning collaboration, cutting‑edge models, and a vibrant global community.*

---  

### 🌟 Who We Are  
Hugging Face is the leading hub where researchers, engineers, and creators come together to **create, discover, and collaborate on machine‑learning (ML) assets**. Our platform hosts **2 M+ models**, **500 k+ datasets**, and **1 M+ applications (Spaces)**, making it the world’s most extensive open‑source AI ecosystem.

### 🚀 Mission & Vision  
- **Democratize AI** – empower anyone to build and use state‑of‑the‑art models, regardless of size or budget.  
- **Foster Collaboration** – provide tools that let the global ML community share, improve, and repurpose AI assets together.  
- **Drive Innovation** – accelerate research and product development through easy‑to‑use APIs, hosted inference, and enterprise‑grade solutions.

### 🛠️ Platform Highlights  

| Area | What You’ll Find | Why It Matters |
|------|------------------|----------------|
| **Models** | 2 M+ pre‑trained models (e.g., Mistral‑Large, DeepSeek‑V4.1, NVIDIA‑Nemotron) | Instant access to the latest architectures for NLP, vision, audio, and more. |
| **Datasets** | 500 k+ curated datasets (e.g., NVIDIA SimReady assets, cyber‑security collections) | Ready‑to‑train data with versioning, metadata, and community annotations. |
| **Spaces** | 1 M+ interactive AI apps (video generation, 3‑D asset search, image editing) | Deploy and showcase ML demos in minutes, powered by GPU or server‑less back‑ends. |
| **Buckets** | Secure storage for large‑scale model and dataset artifacts | Scalable, private storage for enterprise and research pipelines. |
| **Inference** | End‑points, providers, and hosted APIs | Turn any model into a production‑ready service with minimal ops overhead. |
| **Enterprise** | Hugging Face PRO, custom support, and dedicated solutions | Tailored SLAs, compliance, and integration assistance for Fortune‑500 customers. |
| **Community Tools** | Discord, Forum, GitHub, Blog, Daily Papers, Learning hub | Continuous learning, networking, and open‑source contribution channels. |

### 👥 Community & Culture  

- **Open‑source at heart** – All core libraries (Transformers, Diffusers, Datasets) are MIT‑licensed and community‑driven.  
- **Inclusive collaboration** – Contributors span academia, startups, and tech giants; everyone can publish, comment, and improve assets.  
- **Transparency & Ethics** – Dedicated sections for AI ethics, responsible AI, and RLHF research.  
- **Learning mindset** – Regular webinars, tutorials, and a “Learn” portal keep users up‑to‑date with the latest advances.  
- **Fun & playful** – Spaces showcase creative experiments (e.g., video‑from‑image generation, 3‑D asset semantic search) that encourage experimentation.

### 🏢 Who Uses Hugging Face?  

- **Tech giants & hardware vendors** – NVIDIA, Baidu, and other leaders host models and run inference on our platform.  
- **Startups & SaaS providers** – Build AI‑powered products faster by leveraging pre‑trained models and hosted endpoints.  
- **Research labs & universities** – Share reproducible experiments, publish datasets, and collaborate on breakthrough papers.  
- **Enterprises** – Finance, healthcare, and media companies adopt our PRO and Enterprise services for secure, scalable AI deployment.

### 💼 Careers & Opportunities  

Hugging Face is constantly expanding its team of **engineers, researchers, product managers, and community advocates**.  
- **Roles**: ML Engineer, Backend Engineer, Data Scientist, Product Designer, Community Manager, Sales Engineer, etc.  
- **What we offer**: Remote‑first work culture, competitive compensation, equity, continuous learning budget, and the chance to shape the future of open‑source AI.  
- **How to apply**: Visit the **Team & Enterprise** section on our website or follow us on **GitHub** and **Discord** for recruitment announcements.

### 📞 Get Involved  

- **Explore models & datasets** – https://huggingface.co/models & https://huggingface.co/datasets  
- **Try AI apps (Spaces)** – https://huggingface.co/spaces  
- **Read the latest insights** – https://huggingface.co/blog  
- **Join the conversation** – Discord, Forum, and GitHub community  
- **Contact sales or support** – Enterprise & PRO pages for tailored solutions  

---  

**Hugging Face** is more than a platform; it’s a global movement to make AI accessible, collaborative, and responsible. Whether you’re a researcher, developer, business leader, or AI enthusiast, you’ll find the tools, data, and community you need to turn ideas into reality.  

*Join us and help shape the next generation of intelligent applications.*

In [29]:
translate_system_prompt = '''
You are given a brochure, I need you to translate it to Japanese.
'''

In [30]:
translate_user_prompt = '''
Given the following brochure,
please translate it to Japanese language.
'''

In [36]:
def create_and_translate_brochure(system_prompt, user_prompt):
    brochure = create_brochure("HuggingFace", "https://huggingface.co")
    user_prompt += brochure

    response = groq.chat.completions.create(
        model=MODEL, 
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    )

    result = response.choices[0].message.content
    display(Markdown(result))


In [37]:
create_and_translate_brochure(translate_system_prompt, translate_user_prompt)

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 11 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your gateway to open‑source machine learning, collaborative AI development, and enterprise‑grade solutions.*

---

## 1. Who We Are  

Hugging Face is the world’s leading **collaboration platform for the machine‑learning community**. We host a vibrant hub where researchers, developers, and enterprises share **2 M+ models**, **500 k+ datasets**, and **1 M+ AI applications (Spaces)**. Our mission is simple: **make AI accessible, reusable, and trustworthy** for everyone.

---

## 2. What We Offer  

| Pillar | Key Features | Why It Matters |
|--------|--------------|----------------|
| **Models** | Browse, download, and fine‑tune 2 M+ community‑contributed models (e.g., Mistral, DeepSeek, NVIDIA Nemotron). | Accelerate product development with state‑of‑the‑art pretrained models. |
| **Datasets** | 500 k+ curated datasets covering vision, language, audio, security, finance, and more. | Reduce data‑engineering overhead and ensure reproducibility. |
| **Spaces** | Deploy interactive demos and AI apps in seconds, powered by GPUs (RTX PRO 6000, Zero MCP). | Showcase prototypes, collect feedback, and democratize AI usage. |
| **Enterprise Solutions** | Hugging Face PRO, Enterprise Support, Inference Endpoints, Storage Buckets, and dedicated Inference Providers. | Secure, scalable, and compliant AI infrastructure for large organizations. |
| **Community & Learning** | Discord, Forum, Blog, Daily Papers, Hardware guides, and the “Learn” hub. | Stay at the cutting edge, connect with peers, and upskill continuously. |

---

## 3. Company Culture  

- **Open‑Source First** – Every model, dataset, and Space is openly shareable; contributions are celebrated.  
- **Community‑Driven** – Hundreds of thousands of collaborators worldwide shape the roadmap through feedback, pull requests, and community events.  
- **Inclusive & Diverse** – We foster an environment where people from all backgrounds feel welcomed to build and learn together.  
- **Transparency & Ethics** – Clear model cards, licensing, and responsible‑AI guidelines help users understand limitations and risks.  
- **Innovation at Speed** – Continuous releases (e.g., weekly trending models) keep the ecosystem fresh and forward‑looking.

---

## 4. Customers & Partners  

- **Tech Leaders** – Companies such as NVIDIA, Baidu, and DeepSeek host flagship models on our hub.  
- **Enterprises** – Organizations leverage Hugging Face PRO and Enterprise Support for production‑grade inference, data governance, and secure model hosting.  
- **Start‑ups & Researchers** – Over 1 M+ Spaces demonstrate use‑cases ranging from 3‑D asset search to video generation from text prompts.  
- **Academic & Government** – Researchers use our datasets (e.g., cyber‑security, finance tick data) for reproducible science and policy analysis.  

Our **Inference Providers** network lets any cloud or edge provider deliver low‑latency AI services directly from the Hub, expanding reach for customers worldwide.

---

## 5. Careers & Opportunities  

Hugging Face is constantly growing its **Team & Enterprise** division. We look for talent who:

- **Love open source** and want to shape the future of AI collaboration.  
- **Thrive in a distributed, community‑centric environment** (remote‑first, flexible work).  
- **Bring expertise** in machine learning, data engineering, product design, dev‑ops, security, or community management.  

**How to apply** – Visit the **Team & Enterprise** page on our website, explore current openings, and submit your résumé through the portal.  

**Internships & Fellowships** – We partner with universities and research labs, offering hands‑on projects on real‑world AI systems.

---

## 6. Get Started  

- **Explore AI Apps** – Jump into the **Spaces** gallery and interact with cutting‑edge demos.  
- **Browse Models** – Search the Hub for the latest models (e.g., Mistral‑Large‑4‑1T, DeepSeek‑V4.1‑Flash).  
- **Download Datasets** – Leverage high‑quality data for training, evaluation, or research.  
- **Join the Community** – Sign up, contribute, or start a discussion on Discord, the Forum, or GitHub.  

---

## 7. Contact & Resources  

- **Website:** https://huggingface.co  
- **Docs & API:** Comprehensive guides for model integration, dataset loading, and inference endpoints.  
- **Support:** Enterprise Support plans, PRO subscription, and a dedicated help center.  
- **Brand Assets:** Official logos, color palettes, and visual guidelines are available for partners and media.  

*Hugging Face – empowering a global AI community to create, share, and innovate together.*

**Hugging Face – 未来を創る AI コミュニティ**  
*オープンソース機械学習、協働 AI 開発、エンタープライズ向けソリューションへのゲートウェイ。*

---

## 1. 私たちについて  

Hugging Face は、機械学習コミュニティ向けの **世界最大規模の協働プラットフォーム** です。研究者・開発者・企業が **200 万以上のモデル**、**50 万以上のデータセット**、そして **100 万以上の AI アプリケーション（Spaces）** を共有できる活気あるハブを提供しています。私たちのミッションはシンプルです。**すべての人に AI を身近に、再利用しやすく、信頼できるものにする**ことです。

---

## 2. 提供サービス  

| Pillar（柱） | 主な機能 | 重要性 |
|--------------|----------|--------|
| **Models（モデル）** | 2 M+ のコミュニティ提供モデル（例：Mistral、DeepSeek、NVIDIA Nemotron）を検索・ダウンロード・ファインチューニング | 最新の事前学習モデルで製品開発を加速 |
| **Datasets（データセット）** | 50 万以上のキュレーション済みデータセット（画像、自然言語、音声、セキュリティ、金融など） | データエンジニアリングの手間を削減し、再現性を確保 |
| **Spaces（スペース）** | GPU（RTX PRO 6000、Zero MCP）で数秒でインタラクティブデモや AI アプリをデプロイ | プロトタイプを公開し、フィードバックを集め、AI の民主化を実現 |
| **Enterprise Solutions（エンタープライズソリューション）** | Hugging Face PRO、Enterprise Support、Inference Endpoints、Storage Buckets、専用 Inference Provider など | 大規模組織向けに安全・スケーラブル・コンプライアンス対応の AI 基盤を提供 |
| **Community & Learning（コミュニティ & 学習）** | Discord、Forum、Blog、Daily Papers、ハードウェアガイド、Learn ハブ | 最先端情報に触れ、仲間と繋がり、継続的にスキルを向上 |

---

## 3. 企業文化  

- **オープンソース第一** – すべてのモデル・データセット・Space はオープンに共有でき、貢献は歓迎されます。  
- **コミュニティ主導** – 世界中の数十万の協働者がフィードバック、プルリクエスト、イベントを通じてロードマップを形作ります。  
- **包括性と多様性** – すべての背景を持つ人々が歓迎され、共に学び・創造できる環境を育みます。  
- **透明性と倫理** – 明確なモデルカード、ライセンス表記、Responsible‑AI ガイドラインにより、利用者は制限やリスクを理解できます。  
- **スピード感のあるイノベーション** – 週次でトレンドモデルをリリースするなど、エコシステムを常に新鮮かつ先進的に保ちます。

---

## 4. 顧客・パートナー  

- **テックリーダー** – NVIDIA、Baidu、DeepSeek などがフラッグシップモデルを当社ハブで公開。  
- **エンタープライズ** – 大企業は Hugging Face PRO と Enterprise Support を活用し、プロダクション向け推論、データガバナンス、セキュアなモデルホスティングを実現。  
- **スタートアップ・研究者** – 1 M+ 超の Spaces が、3‑D アセット検索からテキストプロンプトによる動画生成まで多様なユースケースを示しています。  
- **学術・政府機関** – サイバーセキュリティや金融ティックデータ等のデータセットを用いて、再現性のある研究や政策分析に活用。  

**Inference Providers** ネットワークにより、クラウドやエッジプロバイダーがハブから直接低遅延 AI サービスを提供でき、世界中の顧客へのリーチを拡大しています。

---

## 5. キャリアと機会  

Hugging Face は **Team & Enterprise** 部門を常に拡大中です。求める人物像は次の通りです。

- **オープンソースが好き** で、AI 協働の未来を創りたい方。  
- **分散型・コミュニティ中心** の環境（リモートファースト、柔軟な働き方）で活躍できる方。  
- **機械学習、データエンジニアリング、プロダクトデザイン、DevOps、セキュリティ、コミュニティマネジメント** などの専門知識を有する方。  

**応募方法** – 公式サイトの **Team & Enterprise** ページから募集職種を確認し、ポータル経由で履歴書を提出してください。  

**インターン・フェローシップ** – 大学や研究ラボと提携し、実践的な AI プロジェクトに参加できる機会を提供しています。

---

## 6. はじめ方  

- **AI アプリを体験** – **Spaces** ギャラリーで最新デモを操作してみましょう。  
- **モデルを検索** – Hub で最新モデル（例：Mistral‑Large‑4‑1T、DeepSeek‑V4.1‑Flash）を探してダウンロード。  
- **データセットを取得** – 高品質データを訓練・評価・研究に活用。  
- **コミュニティに参加** – サインアップ後、Discord、Forum、GitHub で議論や貢献を開始。  

---

## 7. お問い合わせ・リソース  

- **Webサイト:** https://huggingface.co  
- **Docs & API:** モデル統合、データセットロード、推論エンドポイント向けの包括的ガイド。  
- **サポート:** Enterprise Support プラン、PRO サブスクリプション、専用ヘルプセンター。  
- **ブランド資産:** ロゴ、カラーパレット、ビジュアルガイドラインはパートナーやメディア向けに提供しています。  

*Hugging Face – 世界中の AI コミュニティが共に創造・共有・イノベーションを実現できるよう支援します。*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from Groq,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 8 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Hugging Face – The AI Community Building the Future  

**Tagline:** *Create, discover, and collaborate on machine‑learning better.*  

---

### 1. Who We Are  

Hugging Face is the world’s leading open‑source hub for **models, datasets, and AI applications**.  Our platform unites researchers, engineers, startups, and Fortune‑500 enterprises in a single, collaborative ecosystem where ideas turn into production‑ready solutions at speed.

- **Mission:** Democratise AI by providing a free, transparent, and community‑driven marketplace for ML assets.  
- **Vision:** A future where every person and organization can harness state‑of‑the‑art AI safely and responsibly.  

---

### 2. Core Products & Services  

| Offering | What It Does | Typical Users |
|----------|--------------|---------------|
| **Models Hub** | 2 M+ pre‑trained models (LLMs, vision, audio, multilingual, etc.) with versioning, licensing, and community ratings. | Researchers, developers, product teams |
| **Datasets Hub** | 500 k+ curated datasets, searchable by task, language, size, and licensing. | Data scientists, academia, data engineers |
| **Spaces** | Ready‑to‑run AI demos & apps (hosted on GPU/CPU) – no infrastructure needed. | Creators, educators, startups |
| **Inference Endpoints** | Scalable, secure API serving for any model on the Hub – with pay‑as‑you‑go or enterprise contracts. | SaaS providers, enterprises, developers |
| **Buckets (Storage)** | Private, versioned storage for large files, model weights, or proprietary data. | Companies with compliance or privacy requirements |
| **Enterprise & PRO** | Dedicated support, SSO, on‑prem deployment, compliance tools, and advanced analytics. | Large organizations, regulated industries |
| **HuggingChat** | Conversational AI interface built on top of community LLMs, showcasing open‑source chat capabilities. | End‑users, developers exploring chat‑based products |
| **Hardware & Tools** | Optimised runtimes for RTX, Zero‑MCP, and other accelerators; integration with GitHub, Discord, and CI pipelines. | DevOps, ML engineers |

---

### 3. Community & Culture  

- **Open‑source at heart** – Every model, dataset, and Space is publicly visible, forkable, and searchable.  
- **Collaboration first** – Users contribute via pull‑requests, discussions, and “Collections” that group related assets.  
- **Inclusivity & Diversity** – A global community spanning 150+ countries; multilingual support and accessible documentation.  
- **Transparency** – Clear licensing, model cards, and data‑cards that describe provenance, intended use, and limitations.  
- **Learning & Growth** – Free tutorials, “Learn” pathways, daily research paper digests, and an active Discord & Forum for real‑time help.  

---

### 4. Customers & Partners  

Hugging Face powers AI across many sectors. Notable users and collaborators include:

- **Technology giants** – NVIDIA, Baidu, DeepSeek, Mistral AI, Cohere Labs  
- **Finance & Trading** – Venvoo (China A‑Share order‑book data)  
- **Creative & Media** – Companies leveraging image‑to‑video generation and 3‑D asset search (e.g., SimReady Central)  
- **Enterprise adopters** – Fortune‑500 firms using **Inference Endpoints** and **Enterprise Support** for secure, scalable deployments  

These customers trust Hugging Face for:

- Rapid prototyping → production pipelines  
- Secure, compliant model serving  
- Access to the latest research without building from scratch  

---

### 5. Careers & Opportunities  

Hugging Face is constantly expanding its talent pool. We look for people who:

- **Live the open‑source ethos** – contribute to public repos, engage with the community.  
- **Blend research with product** – turn cutting‑edge papers into reliable services.  
- **Champion responsible AI** – embed safety, bias‑mitigation, and privacy into every product.  

Typical roles include:

- Machine‑Learning Engineers & Researchers  
- Software Engineers (frontend, backend, DevOps)  
- Product & Growth Managers  
- Community & Developer Relations  
- Security, Compliance, & Trust & Safety specialists  

**How to apply:**  
Visit the **Team & Enterprise** section on the Hugging Face website, or follow our open positions on GitHub and LinkedIn.  

---

### 6. Get Involved  

- **Explore 2 M+ models** – https://huggingface.co/models  
- **Try an AI app** – https://huggingface.co/spaces  
- **Join the conversation** – Discord, Forum, and weekly **Daily Papers** newsletter.  
- **Download brand assets** – official logos, colors, and guidelines for co‑branding (HF Colors: #FFD21E, #FF9D00, #6B7280).  

---

### 7. Contact  

- **Website:** https://huggingface.co  
- **Support:** Enterprise Support portal (for paid plans)  
- **Community:** Discord, Forum, GitHub (open‑source contributions)  

---

**Hugging Face – Empowering the world to build the next generation of AI, together.**

In [24]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 9 relevant links


# Hugging Face – The AI Community Building the Future  

## 📌 Who We Are  
Hugging Face is the leading open‑source collaboration platform for the global machine‑learning community. Our mission is to democratize artificial intelligence by making models, datasets, and AI‑powered applications discoverable, reusable, and easy to deploy. With more than **2 million models**, **500 k+ datasets**, and **1 million+ interactive Spaces**, we’re the central hub where researchers, developers, enterprises, and hobbyists co‑create the next generation of intelligent systems.  

---

## 🌐 Core Products & Services  

| Offering | What It Does | Why It Matters |
|----------|--------------|----------------|
| **Models Hub** | Browse, share, and version‑control over 2 M+ pretrained models (LLMs, vision, audio, etc.). | Accelerates development by reusing state‑of‑the‑art models. |
| **Datasets Hub** | Access 500 k+ curated datasets with built‑in versioning and licensing. | Reduces data‑collection overhead and ensures reproducibility. |
| **Spaces** | Deploy interactive demos and apps (e.g., image generation, semantic search) with a single click. | Lets anyone showcase AI capabilities without heavy engineering. |
| **Inference Endpoints & Providers** | Scalable, secure API endpoints for production‑grade inference; partners with cloud providers. | Guarantees low‑latency, reliable AI services for business workloads. |
| **Enterprise Solutions** | Private model hosting, compliance controls, dedicated support, and custom SLAs. | Enables large organisations to adopt AI responsibly at scale. |
| **Hugging Face PRO & Support** | Premium tooling, priority assistance, and advanced collaboration features. | Helps power users and teams stay ahead of the curve. |
| **Buckets (Storage)** | Secure storage for large model and dataset files, integrated with the Hub. | Simplifies data management and sharing across teams. |
| **Community & Learning** | Discord, Forum, Blog, Daily Papers, and Learn resources. | Fosters knowledge exchange and continuous skill growth. |

---

## 🤝 Community & Culture  

- **Open Collaboration** – Every model, dataset, and Space is versioned publicly, encouraging peer review, reproducibility, and collective improvement.  
- **Inclusivity** – Our platform supports dozens of languages and offers accessibility tools (e.g., HuggingChat, multilingual docs).  
- **Transparency** – Clear licensing, model cards, and dataset documentation empower responsible AI use.  
- **Innovation‑First** – We host cutting‑edge research from leaders like NVIDIA, DeepSeek, Mistral, Baidu, and many independent contributors.  
- **Playful Identity** – The friendly “🤗” emoji and vibrant brand colors (#FFD21E, #FF9D00, #6B7280) reflect a welcoming, community‑driven spirit.  

---

## 📈 Customers & Partners  

- **Tech Leaders** – NVIDIA, Baidu, DeepSeek, Mistral AI publish flagship models on our Hub.  
- **Enterprises** – Companies leverage Hugging Face Enterprise for secure model deployment and compliance.  
- **Research Labs & Universities** – Use the Hub for open‑source research, data sharing, and reproducible experiments.  
- **Developers & Creators** – Build and share interactive AI applications through Spaces, reaching millions of users.  

---

## 👩‍💼 Careers & Opportunities  

Hugging Face is constantly expanding its team across engineering, research, product, design, and community roles.  

- **Roles** – Machine‑learning engineers, data scientists, product managers, dev‑ops, UX designers, community managers, and more.  
- **Work Environment** – Remote‑first, flexible hours, collaborative code reviews, and an emphasis on learning (internal workshops, hack weeks).  
- **Why Join?** – Contribute to a platform that powers the world’s most advanced AI models, work alongside leading AI researchers, and help shape an open‑source ecosystem that impacts millions.  

*Explore current openings on the “Team & Enterprise” page or follow us on Discord and our blog for updates.*  

---

## 🚀 How to Get Started  

1. **Explore** – Visit the Hub, browse the 2 M+ models or 500 k+ datasets.  
2. **Create** – Sign up for a free account, upload your own model or dataset, and add a clear model card.  
3. **Deploy** – Turn any model into an interactive Space or launch an Inference Endpoint in minutes.  
4. **Collaborate** – Join the community forums, Discord, or contribute to open‑source projects.  
5. **Scale** – Upgrade to Hugging Face PRO or Enterprise for private hosting, compliance, and dedicated support.  

---

## 📬 Contact & Resources  

- **Website:** https://huggingface.co  
- **Documentation:** Docs, API references, and tutorials (Learn section).  
- **Community:** Discord, Forum, Blog, Daily Papers.  
- **Brand Assets:** Official logos (SVG/PNG/AI) and color palette available for partners.  

*Join the AI revolution. Build, share, and deploy smarter with Hugging Face.*  

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>